# Site2Tools — control base vs LoRA en T4 (Colab)

Una sola celda controladora: comprueba la GPU, clona `dbqp01/ai-test` y repite en T4 la tabla de control que se midio en la MI300X (exactitud de accion, modelo base vs adapter v4, sobre las tres distribuciones). Si la tabla no se sostiene en otra GPU, lo que dependia del hardware era la medida, no el modelo.

No reinstala torch (romperia el CUDA de Colab) y no sube nada: solo lee el repo publico.

Coste: 8-15 min con `LIMITE=120` por distribucion y modelo.

In [ ]:
import subprocess, sys, time, re

def correr(cmd, etiqueta):
    t0 = time.time()
    r = subprocess.run(cmd, capture_output=True, text=True)
    salida = (r.stdout or "") + (r.stderr or "")
    print(f"--- {etiqueta} ({time.time()-t0:.0f}s, rc={r.returncode})", flush=True)
    return salida

print("== entorno ==", flush=True)
print(correr(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], "gpu"))
import torch, transformers, peft
print("torch", torch.__version__, "cuda", torch.cuda.is_available(),
      "transformers", transformers.__version__, "peft", peft.__version__, flush=True)

print("== clonando el repo ==", flush=True)
subprocess.run(["git", "clone", "-q", "https://github.com/dbqp01/ai-test.git", "/content/ai-test"])
subprocess.run(["git", "-C", "/content/ai-test", "log", "--oneline", "-1"])

ADAPTER = "/content/ai-test/artifacts/site2tools-student-17b-v4"
SPLITS = {"sintetica v3": "data/v3.valid.jsonl",
          "mind2web plano": "data/mind2web.valid.jsonl",
          "m2w2 realista": "data/m2w2.valid.jsonl"}
LIMITE = 120
tabla = {}
for nombre, split in SPLITS.items():
    for etiqueta, adapter in (("BASE", None), ("v4", ADAPTER)):
        cmd = [sys.executable, "eval_action_acc.py", "--valid", split, "--limit", str(LIMITE)]
        if adapter:
            cmd += ["--adapter", adapter]
        salida = correr(cmd, f"{nombre} / {etiqueta}")
        m = re.findall(r"([01]\.\d{3})", salida)
        tabla[(nombre, etiqueta)] = (m[-1] if m else "sin-lectura", salida[-260:])

print("\n===== CONTROL base vs LoRA en T4 (exactitud de accion) =====", flush=True)
for nombre in SPLITS:
    b, v = tabla[(nombre, "BASE")][0], tabla[(nombre, "v4")][0]
    print(f"  {nombre:16s} base={b}  v4={v}", flush=True)
print("(referencia MI300X: base 0.322/0.191/0.068  vs  LoRA 0.753/0.586/0.781)", flush=True)
print("\n--- ultimas lineas de cada eval, por si algun numero no se leyo ---", flush=True)
for k, (_, cruce) in tabla.items():
    print(f"[{k[0]} {k[1]}] ...{cruce[-200:]}", flush=True)
